# Validation Test — Simple RAG Pipeline

End-to-end test of `validate_env()` with a real Corpora backend.
Installs from the `feat/env-validation` branch.

In [2]:
!pip install "cgft[dev] @ git+https://github.com/cgftinc/cgft.git@feat/env-validation"

  Cloning https://github.com/cgftinc/cgft.git (to revision feat/env-validation) to /private/var/folders/8w/8rh15bd90x7bn744lzw9xrhm0000gn/T/pip-install-9rz4le86/cgft_59cc5edf61814cf7946f76421c0e46aa
  Running command git clone --filter=blob:none --quiet https://github.com/cgftinc/cgft.git /private/var/folders/8w/8rh15bd90x7bn744lzw9xrhm0000gn/T/pip-install-9rz4le86/cgft_59cc5edf61814cf7946f76421c0e46aa
  Running command git checkout -b feat/env-validation --track origin/feat/env-validation
  Switched to a new branch 'feat/env-validation'
  branch 'feat/env-validation' set up to track 'origin/feat/env-validation'.
  Resolved https://github.com/cgftinc/cgft.git to commit 23fc656e844b56b03c9d56edd3fb9f778ad7b291
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for cgft: filename=cgft-0.1.0-py3-none-any.whl size=251894 sha256=771123b97a51f57f78768498c3ec737e45423e9af7519327363e71526d8506ae

In [3]:
!rm -rf posthog.com
!git clone --depth 1 --filter=blob:none --sparse https://github.com/PostHog/posthog.com.git
%cd posthog.com
!git sparse-checkout set contents/docs
%cd ..

Cloning into 'posthog.com'...
remote: Enumerating objects: 995, done.
remote: Counting objects: 100% (995/995), done.
remote: Compressing objects: 100% (548/548), done.
remote: Total 995 (delta 0), reused 783 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (995/995), 222.97 KiB | 6.03 MiB/s, done.
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 33 (delta 1), reused 14 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 457.24 KiB | 3.54 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (33/33), done.
/Users/thariqridha/Projects/cgft_projects/notebooks/internal/posthog.com
remote: Enumerating objects: 1401, done.
remote: Counting objects: 100% (1401/1401), done.
remote: Compressing objects: 100% (1053/1053), done.
remote: Total 1401 (delta 405), reused 697 (delta 344), pack-reused 0 (from 0)
Receiving objects: 100% (1401/1401), 1.46 MiB | 6.55 

In [4]:
# ── Credentials (fill these in) ───────────────────────────────────
API_KEY = ""          # CGFT platform key — https://app.cgft.io/account/api-keys
BASE_URL = "https://app.cgft.io"

LLM_API_KEY = ""      # Azure OpenAI / OpenAI key for QA generation
LLM_BASE_URL = "https://judge-elsa.services.ai.azure.com/openai/v1/"
LLM_MODEL = "grok-4-fast-reasoning"

JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "grok-4-fast-reasoning"

# ── Corpus ────────────────────────────────────────────────────────
DOCS_PATH = "./posthog.com/contents/docs/"
CORPUS_NAME = "posthog-validation-test"
CORPUS_DESCRIPTION = "PostHog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# ── QA Generation ────────────────────────────────────────────────
TOTAL_SAMPLES = 20  # small for testing
OUTPUT_DIR = "outputs/validation_test"
EXPERIMENT_NAME = "validation-test-v1" 

In [5]:
from cgft.utils import ensure_safe_python_version
ensure_safe_python_version()

## Step 1: Upload Corpus

In [6]:
from cgft.corpus.corpora.source import CorporaChunkSource

source = CorporaChunkSource(
    api_key=API_KEY,
    corpus_name=CORPUS_NAME,
    base_url=BASE_URL,
)
source.populate_from_folder(DOCS_PATH)
print(f"Corpus ID: {source.corpus_id}")
print(f"Chunks: {source._get_total_count()}")

Chunking documents from ./posthog.com/contents/docs/...
ChunkCollection Summary
Total chunks: 3532
Total files: 1271

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1149 chars

File Structure:
├── data-warehouse/
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── reddit-ads.mdx (1 chunks)
│   │   ├── s3.mdx (1 chunks)
│   │   ├── temporal.mdx (1 chunks)
│   │   ├── klaviyo.mdx (1 chunks)
│   │       ... 37 more files
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── sql/
│   │   ├── variables.mdx (2 chunks)
│   │   ├── index.mdx (8 chunks)
│   │   ├── data-access.mdx (3 chunks)
│   │   └── useful-functions.mdx (7 chunks)
│   ├── under-the-hood.md (1 chunks)
│   ├── cutting-costs.mdx (3 chunks)
│   ├── join.mdx (2 chunks)
│   ├── run-sql-mcp.mdx (4 chunks)
│       ... 8 more files
├── advanced/
│   ├── proxy/
│   │  

Uploading chunks:   0%|          | 0/36 [00:00<?, ?batch/s]


Upload complete! Inserted: 3532
Corpus ID: d7369cf7-25c5-4418-a6c3-7bc51d83f211


AttributeError: 'CorporaChunkSource' object has no attribute '_get_total_count'

## Step 2: Generate Synthetic QA

In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LLMDirectGenerationConfig,
    OutputConfig,
    PlatformConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

cfg = CgftPipelineConfig(
    platform=PlatformConfig(
        api_key=API_KEY,
        base_url=BASE_URL,
        llm_api_key=LLM_API_KEY,
        llm_base_url=LLM_BASE_URL,
    ),
    corpus=CorpusConfig(corpus_name=CORPUS_NAME, min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        model=LLM_MODEL,
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(model=LLM_MODEL, max_concurrent=4),
    ),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(judge_model=LLM_MODEL, max_concurrent=4),
        grounding_llm=GroundingLLMFilterConfig(judge_model=LLM_MODEL, max_concurrent=4),
    ),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
print(f"Generated: {len(train_data)} train / {len(eval_data)} eval")

## Step 3: Validate Environment

This is the new `validate_env()` — runs a full simulated rollout locally before submitting a training job.

In [ ]:
from cgft.corpus.corpora.search import CorporaSearch
from cgft.envs.search_env import SearchEnv
from cgft.trainer.validation import validate_env

search = CorporaSearch(
    api_key=API_KEY,
    corpus_name=CORPUS_NAME,
    base_url=BASE_URL,
    corpus_id=source.corpus_id,
)

# Run full validation — simulated rollout, tool calls, reward, pickle
validate_env(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
)

## Step 4: Train (dry run)

`validate_env=True` is the default — `train()` now runs the same validation automatically before uploading.

In [ ]:
import cgft
from cgft.trainer.pipeline import train

result = train(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="validation-test",
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    experiment_name=EXPERIMENT_NAME,
    dry_run=True,  # validate everything, don't launch
)

print(f"\nDry run result: {result}")

## Step 5: Launch Training (optional)

Uncomment to actually launch. Only do this after dry_run passes.

In [ ]:
# experiment_id = train(
#     env_class=SearchEnv,
#     env_args={
#         "search": search,
#         "judge_base_url": JUDGE_BASE_URL,
#         "judge_api_key": JUDGE_API_KEY,
#         "judge_model": JUDGE_MODEL,
#     },
#     train_dataset=train_data,
#     eval_dataset=eval_data,
#     prefix="validation-test",
#     api_key=API_KEY,
#     base_url=BASE_URL,
#     local_modules=[cgft],
#     experiment_name=EXPERIMENT_NAME,
# )
# print(f"Experiment launched: {experiment_id}")